# Buyer GPT B1, B2 and B2f6: the same prompts, side by side

**Sell Anything, notebook 6.** B2 is the second version of the buyer model. It is trained the same way as B1
(notebook 4), but on more sales: the 100 old ones plus the 150 new sales of seeds 2200–2349, written under the
quality checks, in prompt format 5 (no budget on the card). **B2f6** is B2 again, trained on exactly the same 250
sales, but in prompt format 6: every drawable thing, every buyer persona and the kind of animal get a token of their
own (section 10). This notebook only reads the results of the held-out evaluation and of the experiments; it does not
run any model, so it opens in seconds on any machine.

**How the models are compared.** `tools/eval_buyer_b1.py` gives every model **exactly the same prompts**:
the same buyer card, the same story so far and the same seller answer. Each model writes one line (an opening, a
question, a reaction or a scene), sampled the way the game samples (temperature 0.7, top-k 50, a fixed seed for
every prompt), and the line is checked by the program rules every training sale passed. So any difference between
the columns comes from the model. (B2f6 reads the same prompts written in format 6; section 2 checks that they say
exactly the same thing.)

⚠️ This is not two live sales. Every prompt is cut from a written sale, so the story before each line is the written
one, not what the model said earlier. That makes the comparison fair line by line, but it cannot show what happens
when a model's own mistake leads to the next one.

In [1]:
import hashlib
import json
import math
import re
import sys
from collections import Counter
from pathlib import Path

ML = Path.cwd().parent
sys.path.insert(0, str(ML))
from sellanything_ml.buyers.cat_market import MAX_NEW

EVAL = ML / "runs/sft_b2/eval"
F6 = ML / "runs/sft_b2f6"
MODELS = ("B1", "B2", "B2f6")
RUNS = {"old": (ML / "runs/sft_b1/eval_v2_no_budget", EVAL / "b2_val_old", F6 / "eval/b2f6_val_old"),
        "new": (EVAL / "b1_val_new", EVAL / "b2_val_new", F6 / "eval/b2f6_val_new")}   # (B1, B2, B2f6) for each set
THING = re.compile(r"<\|thing:([^|]+)\|>")       # format 6 writes a scene's things as tokens


def scene_things(text):
    """The things at the start of a scene: "a, b, c | line" (formats 4 and 5) or three thing tokens (format 6)."""
    head = text.split(" | ")[0] if " | " in text else text.split("|")[0]
    return THING.findall(head) or [t.strip() for t in head.split(",")]


def report(stem):
    return json.load(open(f"{stem}.json"))["tasks"]


def samples(stem):
    """The example lines the evaluation keeps: the first 12 of each task, in the order of the validation file."""
    out, task, cur = [], None, None
    for line in open(f"{stem}_samples.md").read().splitlines():
        if line.startswith("## "):
            task = line[3:].strip()
        elif m := re.match(r"\*\*(\w+)\*\* \(sale (\d+)\) — (.*)", line):
            cur = {"task": task, "name": m[1], "seed": m[2], "story": m[3]}
            out.append(cur)
        elif cur is not None:
            for key in ("sale", "model", "checks"):
                if line.startswith(f"- {key}: "):
                    cur[key] = line[len(key) + 4:]
    return out


def pairs(which):
    """B1's, B2's and B2f6's line for every kept example; the runs read the same prompts in the same order (B2f6's
    are written in format 6, so for it only the task, the buyer and the sale are compared)."""
    a, b, c = (samples(stem) for stem in RUNS[which])
    for x, y, z in zip(a, b, c, strict=True):
        assert (x["task"], x["name"], x["seed"], x["story"]) == (y["task"], y["name"], y["seed"], y["story"])
        assert (x["task"], x["name"], x["seed"]) == (z["task"], z["name"], z["seed"])
    return list(zip(a, b, c))


def show(which, task):
    for k, rows in enumerate([p for p in pairs(which) if p[0]["task"] == task], 1):
        x = rows[0]
        story = re.sub(r"<\|\w+\|>", "", x["story"]).split(" / ")
        print(f"[{k}] {x['name']} (sale {x['seed']})")
        for part in story:
            print("    " + part)
        print("  sale:", x["sale"])
        for label, row in zip(MODELS, rows):
            print(f"  {label + ':':6}", THING.sub(r"‹\1›", row["model"]))   # ‹pencil› = the token <|thing:pencil|>
            print("        check:", row["checks"])
        print()

## 1. Training B2 and B2f6

Same recipe as B1: start from `pretrain_large` (71M parameters), 3 epochs, lr 3e-4, loss on the target only. B2 ran
on one Modal A100 (`tools/modal_sft_b2.py`), batch 16, 4,844 steps; B2f6 ran the same steps with the same batch on
this Mac's GPU.

In [2]:
for name in ("sft_b1", "sft_b2", "sft_b2f6"):
    rows = [json.loads(line) for line in open(ML / "runs" / name / "metrics.jsonl")]
    best = min(rows, key=lambda r: r["val_loss"])
    print(f"{name}: {len(rows)} checks, best val loss {best['val_loss']:.4f} at step {best['step']}, "
          f"last {rows[-1]['val_loss']:.4f} at step {rows[-1]['step']}")
    print("   ", " ".join(f"{r['step']}:{r['val_loss']:.3f}" for r in rows))

sft_b1: 10 checks, best val loss 0.9063 at step 3733, last 0.9063 at step 3733
    400:1.236 800:1.143 1200:1.072 1600:1.052 2000:0.976 2400:0.931 2800:0.938 3200:0.930 3600:0.908 3733:0.906
sft_b2: 10 checks, best val loss 1.3156 at step 3000, last 1.3222 at step 4844
    500:1.687 1000:1.555 1500:1.464 2000:1.424 2500:1.369 3000:1.316 3500:1.343 4000:1.336 4500:1.324 4844:1.322
sft_b2f6: 10 checks, best val loss 1.3339 at step 3000, last 1.3418 at step 4844
    500:1.710 1000:1.570 1500:1.475 2000:1.444 2500:1.394 3000:1.334 3500:1.363 4000:1.357 4500:1.345 4844:1.342


B1's validation loss was still falling at its last step (0.906). B2's lowest validation loss is 1.316 at step 3000, about 1.9 epochs in; after that it goes up to 1.343 and ends at 1.322, so the second half of training did not help and `best.pt` is the step 3000 weights. The two numbers cannot be compared with each other: B2's validation set adds 17 new sales, whose lines are longer and use many more different words, so they are simply harder to predict. That is why the rest of this notebook scores both models on the same prompts instead.

B2f6 follows the same curve: its lowest validation loss is 1.334 at step 3000, then 1.363 at step 3500 and 1.342 at the end, while the training loss falls to about 0.6, so the model is memorising the training sales from about two epochs on. Its 1.334 cannot be compared with B2's 1.316 either: a thing is now one token instead of two or three, so the scenes have fewer, harder tokens to predict and the average per token changes.

## 2. The two validation sets

Both models are scored on sales that **neither** trained on. The old set is the 10 validation sales B1 was already
scored on; the new set is 17 sales from the new batch (seeds 2200–2349) that were held out of B2's training.

In [3]:
train_b2 = {json.loads(line)["seed"] for line in open(ML / "data/processed/buyers_b2/train.jsonl")}
train_b1 = {json.loads(line)["seed"] for line in open(ML / "data/processed/buyers_b1/train.jsonl")}
print(f"training sales: B1 {len(train_b1)}, B2 {len(train_b2)}")
for which in ("old", "new"):
    rows = [json.loads(line) for line in open(ML / f"runs/eval_sets_b2/val_{which}_format5.jsonl")]
    seeds = {r["seed"] for r in rows}
    print(f"{which} set: {len(seeds)} sales, {len(rows)} prompts {dict(Counter(r['task'] for r in rows))}, "
          f"seeds {min(seeds)}–{max(seeds)}, in B1 training {len(seeds & train_b1)}, in B2 training {len(seeds & train_b2)}")

training sales: B1 90, B2 223
old set: 10 sales, 1195 prompts {'open': 100, 'ask': 98, 'react': 908, 'imagine': 89}, seeds 2004–2067, in B1 training 0, in B2 training 0
new set: 17 sales, 1998 prompts {'open': 170, 'ask': 163, 'react': 1513, 'imagine': 152}, seeds 2204–2343, in B1 training 0, in B2 training 0


B2 trained on 223 sales, B1 on 90. Neither set shares a single sale with either model's training data, so every line below is written for a sale the model has never seen. The new set is bigger (1,998 prompts against 1,195), and nine in ten prompts in both sets are reactions, so the reaction scores rest on the most data.

B2f6 is scored on the same prompts written in format 6. Writing every token back as its words should give the
format-5 prompt and answer letter for letter:

In [4]:
from sellanything_ml.buyers.format import ANIMAL_TOKENS, BUYER_TOKENS

LIST = re.compile(r"((?:<\|thing:[^|]+\|>)+)")


def as_words(text):
    """A format-6 text in format-5 words: three thing tokens -> "a, b, c", a persona token -> "who ...", <|cat|> -> cat."""
    text = LIST.sub(lambda m: ", ".join(THING.findall(m[1])), text)
    for persona, token in BUYER_TOKENS.items():
        text = text.replace(token, "who " + persona)
    for animal, token in ANIMAL_TOKENS.items():
        text = text.replace(token, animal)
    return text


train_f6 = {json.loads(line)["seed"] for line in open(ML / "data/processed/buyers_b2_f6/train.jsonl")}
print("B2f6 trains on the same sales as B2:", train_f6 == train_b2)
for which in ("old", "new"):
    f5 = [json.loads(line) for line in open(ML / f"runs/eval_sets_b2/val_{which}_format5.jsonl")]
    f6 = [json.loads(line) for line in open(ML / f"runs/eval_sets_b2f6/val_{which}_format6.jsonl")]
    same = sum(a["prompt"] == as_words(b["prompt"]) and a["target"] == as_words(b["target"]) for a, b in zip(f5, f6))
    print(f"{which} set: {len(f6)} format-6 prompts, {same} say exactly what the format-5 prompt says")
example = next(json.loads(line) for line in open(ML / "runs/eval_sets_b2f6/val_new_format6.jsonl")
               if json.loads(line)["task"] == "imagine")
print("\nformat 6, start of a card:", example["prompt"][:120])
print("format 6, a written scene:", example["target"])

B2f6 trains on the same sales as B2: True
old set: 1195 format-6 prompts, 1195 say exactly what the format-5 prompt says


new set: 1998 format-6 prompts, 1998 say exactly what the format-5 prompt says

format 6, start of a card: <|item|><|cat|><|end|><|persona|>Chloe, <|buyer:is a student on a tight budget|> | cares about: cheap to keep, few upfro
format 6, a written scene: <|thing:laptop|><|thing:book|><|thing:coffee cup|> | I'd study at my laptop with a secondhand book open while she naps nearby, and my coffee cup keeps me awake through her noisy evenings.<|end|>


B2f6 trains on exactly B2's 223 sales, and every one of the 1,195 + 1,998 format-6 prompts says exactly what the format-5 prompt says once its tokens are written back as words. So the three models answer the same questions, and B2 and B2f6 learned from the same examples: any difference between them comes from the format. The example card shows the persona as one token (`<|buyer:is a student on a tight budget|>`) and the cat as `<|cat|>`; the written scene starts with three thing tokens.

## 3. Scores

Each number is the share of lines that pass a check, over every prompt of the set. "Same direction" means the model
moved its offer the same way (up, down, keep, walk away) as the buyer in the written sale; "mean offer error" is how
far the model's new offer is from the sale's, in dollars.

In [5]:
ROWS = [("open: readable", "open", "readable"), ("open: spoken line passes", "open", "spoken_ok"),
        ("ask: readable", "ask", "readable"), ("ask: not the same words as before", "ask", "new_question"),
        ("ask: new in meaning", "ask", "new_in_meaning"), ("react: readable", "react", "readable"),
        ("react: passes the rules", "react", "rules_ok"), ("react: same direction as the sale", "react", "same_direction"),
        ("react: amount said = offer", "react", "said_equals_offer"), ("react: reasonable move", "react", "reasonable"),
        ("react: mean offer error ($)", "react", "mean_offer_error"),
        ("imagine: readable", "imagine", "readable"), ("imagine: passes the rules", "imagine", "rules_ok")]
for which in ("old", "new"):
    reps = [report(stem) for stem in RUNS[which]]
    print(f"{which} set{'':30}" + "".join(f"{m:>8}" for m in MODELS))
    for label, task, key in ROWS:
        fmt = "{:>8.2f}" if key == "mean_offer_error" else "{:>8.1%}"
        print(f"  {label:36}" + "".join(fmt.format(r[task][key]) for r in reps))
    sale = reps[0]["react"]["withdrawals"]["sale"]
    print(f"  {'react: walk-aways (sale has ' + str(sale) + ')':36}" + "".join(f"{r['react']['withdrawals']['model']:>8}" for r in reps))
    gaps = []                     # the opening offer against the sale's, over the 12 kept openings
    for stem in RUNS[which]:
        opens = [r for r in samples(stem) if r["task"] == "open"]
        gaps.append(sum(abs(int(r["sale"].rsplit("$", 1)[1]) - int(r["model"].rsplit("$", 1)[1])) for r in opens) / len(opens))
    print(f"  {'open: offer off by, 12 kept ($)':36}" + "".join(f"{g:>8.2f}" for g in gaps))
    print()

old set                                    B1      B2    B2f6
  open: readable                        100.0%  100.0%  100.0%
  open: spoken line passes              100.0%  100.0%  100.0%
  ask: readable                         100.0%  100.0%  100.0%
  ask: not the same words as before     100.0%  100.0%  100.0%
  ask: new in meaning                    89.8%   90.8%   92.9%
  react: readable                        99.6%  100.0%   99.9%
  react: passes the rules                90.2%   90.4%   90.0%
  react: same direction as the sale      78.8%   77.5%   76.1%
  react: amount said = offer             99.9%  100.0%  100.0%
  react: reasonable move                 99.9%   99.9%   99.8%
  react: mean offer error ($)             2.52    2.58    2.54
  imagine: readable                     100.0%  100.0%  100.0%
  imagine: passes the rules              70.8%   38.2%   50.6%
  react: walk-aways (sale has 11)           14      10      14
  open: offer off by, 12 kept ($)         8.08   15.83  

**Old set (the style B1 was trained on):** the three models are almost level on reactions (passing the rules 90.2%, 90.4%, 90.0%; same direction as the sale 78.8%, 77.5%, 76.1%; offer off by $2.52, $2.58, $2.54). With about 900 reactions, one standard error of a difference between two models is about 2 points, so none of these gaps is real evidence. B2f6 asks the most questions that are new in meaning (92.9%). Its scenes pass the rules more often than B2's (50.6% against 38.2%), still far below B1 (70.8%).

**New set (the style of the 150 new sales):** B2 is clearly better than B1 at reactions, and B2f6 is close to B2: more reactions pass the rules (93.1% against B2's 92.3% and B1's 86.7%), slightly fewer move the same way as the sale (72.8% against 74.2%, within the 1.6-point standard error), the offer is a little further off ($3.38 against $2.93; B1 $4.62), and B2f6 walks away 22 times where the written buyers walked away 18 times (B2 14, B1 39). Its openings are all readable (B1 97.6%, B2 98.2%).

**Scenes:** on the new set B2f6 passes 14.5%, against B2's 11.9% and B1's 59.9%. Format 6 did not fix the scenes.

The opening offers are only kept for 12 openings per set, so that row is rough: on the old set B1 is closest to the sale ($8.08 off), on the new set B2 ($4.83) and B2f6 ($6.08), with B1 far off ($17.42).

## 4. Which rules the lines break

The evaluation counts the problems the program rules find (the five most common for each task).

In [6]:
for which in ("old", "new"):
    for task in ("react", "imagine"):
        for label, stem in zip(MODELS, RUNS[which]):
            probs = report(stem)[task].get("problems", {})
            print(f"{which} {task} {label} ({sum(probs.values())} counted):")
            for problem, n in probs.items():
                print(f"   {n:4}  {problem}")
        print()

old react B1 (89 counted):
     85  an unchanged offer must name what the seller just said
      1  sign - does not match the offer going from $N to $N
      1  "exactly what i want" gives no reason; say what was just said and what it means for your own life
      1  sign + does not match the offer going from $N to $N
      1  the line says $N but the offer is $N
old react B2 (87 counted):
     83  an unchanged offer must name what the seller just said
      2  sign + does not match the offer going from $N to $N
      1  "exactly what i want" gives no reason; say what was just said and what it means for your own life
      1  sign - does not match the offer going from $N to $N
old react B2f6 (91 counted):
     81  an unchanged offer must name what the seller just said
      7  sign + does not match the offer going from $N to $N
      2  "exactly what i want" gives no reason; say what was just said and what it means for your own life
      1  sign - does not match the offer going from $

Almost every reaction that fails breaks the same rule: the offer stays the same but the line does not name anything the seller just said. On the new set B2 makes this mistake 114 times against B1's 199. For scenes, nearly every failure is a thing that was picked but never used in the line ("the line never names ..."), and B2's list is longer.

B2f6 makes the "unchanged offer that names nothing" mistake about as often as B2 (81 on the old set, 97 on the new set against B2's 83 and 114), but it says a sign that does not match its own new offer more often (8 times on the old set against B2's 3, 6 times on the new set against 1). Its most common scene problem on the new set is one that is not among the other two models' five most common: **"things must be a list of exactly 3 items" (21 times)**, a list that is not three thing tokens (section 10.1; B2 makes it 4 times in the 241 scenes of section 8).

## 5. Every kept line, side by side: the old validation set

The evaluation keeps the first 12 prompts of each task. For each one: the end of the story the buyer sees, what the
buyer in the written sale said, then B1's, B2's and B2f6's line and what the program check says about it. In B2f6's
scenes `‹pencil›` stands for the token `<|thing:pencil|>`.

### Openings

In [7]:
show("old", "open")

[1] Leo (sale 2004)
    (nothing yet)
  sale: I've got plenty of free time to keep a cat entertained. | $50
  B1:    I'm retired and home all day, so I want a cat that likes company. | $70
        check: {"readable": true, "offer": 70, "spoken_ok": true}
  B2:    I'm retired, so I need a cat that likes company and meets my calm needs. | $60
        check: {"readable": true, "offer": 60, "spoken_ok": true}
  B2f6:  Good morning. I'm retired, and I'd love a cat who wants to be around people. | $60
        check: {"readable": true, "offer": 60, "spoken_ok": true}

[2] Ruby (sale 2004)
    (nothing yet)
  sale: I've missed having a cat around since mine passed away last year. | $60
  B1:    I miss my old cat and I want a cat to keep me company. | $70
        check: {"readable": true, "offer": 70, "spoken_ok": true}
  B2:    I lost my old cat last year, and I'm looking for someone to keep me company. | $100
        check: {"readable": true, "offer": 100, "spoken_ok": true}
  B2f6:  I lost m

Both models write a readable opening for all 12 buyers. B1's offers stay closer to the written sale here: the average gap is about $8 for B1 and about $16 for B2, mostly because of B2's large swings (Ruby $100 where the sale has $60, Felix $150 where it has $100, Chloe $40 where it has $65). Some of B2's wording is already off: Aisha needs a cat that "won't scare me" and Sam wants a cat that is "patient with them and a lot of patience".

B2f6's openings are about as close to the sale as B2's (about $13 off on average, against B2's $16 and B1's $8): Chloe at $30 and Mei at $30 where the sale has $65 and $55 pull it down. Its wording breaks in other places: Sam has "two big Pilots at home" (the card says two dogs) and Diego needs a cat that "doesn't be too active at night".

### Questions

In [8]:
show("old", "ask")

[1] Ben (sale 2004)
    Your opening ($75): I'm a bit of a perfectionist when it comes to my pets.
  sale: Can you tell me if the cat has any health issues and if all the vaccinations are up to date?
  B1:    Can you tell me more about the cat's health and if it has any known health issues?
        check: {"readable": true, "new_question": true, "new_in_meaning": true}
  B2:    Can you tell me more about the cat's health history and if it has had all its vaccinations up to date?
        check: {"readable": true, "new_question": true, "new_in_meaning": true}
  B2f6:  Can you tell me more about the cat's health history and if it has had check-ups?
        check: {"readable": true, "new_question": true, "new_in_meaning": true}

[2] Leo (sale 2004)
    Ben asked: Can you tell me if the cat has any health issues and if all the vaccinations are up to date?
    Seller: Yea, shes FIV pos but shes fully vaccinated and had a vet check last month. She's all up to date on her stuff.
    You ($50 -

Every question is readable and new, but B1 asks two that repeat an earlier question in meaning ([5] FIV and other cats, [9] how active she is), and one with a broken word ([8] "heroving nature"). B2's questions repeat less but some ramble or lose their meaning: [2] "well socialized for long years with her", [10] "was it ever my two?", [12] "her grace and if she's been allowed to go in a lot?".

B2f6's questions are readable and mostly fit the card ([5] Aisha asks whether she is friendly with other cats, [7] Chloe whether she is fine alone for long periods), but [3] Diego, who works nights, asks about health instead of nights, [10] Sam calls the cat "he", [12] Leo asks about "her eye for how she looks", and [4] and [9] are counted as repeats in meaning.

### Reactions

In [9]:
show("old", "react")

[1] Leo (sale 2004)
    Your opening ($50): I've got plenty of free time to keep a cat entertained.
    Ben asked: Can you tell me if the cat has any health issues and if all the vaccinations are up to date?
    Seller: Yea, shes FIV pos but shes fully vaccinated and had a vet check last month. She's all up to date on her stuff.
  sale: + | Being FIV positive means she stays indoors, which suits me since I'm home all day, and a vet check last month is reassuring. I'll go up to $55. | $55
  B1:    0 | A vet check last month is something I have time for. Staying at $50. | $50
        check: {"readable": true, "rules_ok": true, "same_direction": false, "withdrew": false, "teacher_withdrew": false, "offer_error": 5, "said_equals_offer": true, "reasonable": true}
  B2:    0 | A vet check last month is good, but I'll hear whether she likes routine care. I'll stay at $50. | $50
        check: {"readable": true, "rules_ok": true, "same_direction": false, "withdrew": false, "teacher_withdrew": 

This set shows the same weakness in both models: they misread the seller. The seller says the cat is **fully vaccinated** and had a vet check last month, and still B1 ([4], [5], [7]) and B2 ([4], [5]) talk about "no vaccinations". Two of B2's mistakes are worse than B1's: [4] Felix, who breeds show cats, lowers to $90 instead of walking away from an FIV-positive cat, and [5] Ben raises his offer while saying the cat is "the opposite of the perfect health I want". B2 also fails the rules twice ([11] "exactly what I want", [12] an unchanged offer that names nothing the seller said); B1 passes all 12 but [9] Chloe's line is broken English. So on this set B2's reactions are not better, which matches the scores.

B2f6 hears this answer better in one place and worse in others. [4] Felix, the breeder, withdraws for the right reason ("FIV positive ... she can never join my breeding program"), which B2 did not do. But [9] Chloe hears "no vaccinations", [5] Ben raises his offer for a cat "in excellent health" without a word about FIV, [8] Aisha raises because her older cat "can get along" with an FIV-positive cat (a risk the written buyer lowers for), and [7] Maya raises because the vaccinations "won't scare her". [12] Ruby's line ("it doesn't change me for me") passes the rules but is broken English.

### Scenes

In [10]:
show("old", "imagine")

[1] Leo (sale 2004)
    Owen asked: Can you tell me how she handles being in a small space and if she's likely to be inactive, since I live in a small apartment?
    Seller: She's pretty laid back and doesn't mind small spaces, so she should do fine in an apartment. She's not super active and likes her quiet time, so she won't take up too much space or cause a ruckus.
    You ($52 -> $52): Not super active fits my calm days at home. Staying at $52.
  sale: chair, radio, clock | She'd keep me company by my chair while the radio plays my old songs, and the clock tells us when it's time for her dinner.
  B1:    chair, teapot, clock | She'd nap by my chair while I pour from my teapot, and I'd set my alarm clock for her breakfast on my lap time.
        check: {"readable": true, "rules_ok": false} — the line mentions alarm clock without picking it; name only the things picked
  B2:    chair, door, pillow | She'd sit on my lap while I read my book and pill when I'm home, and I'd wear my old 

B1 passes 7 of these 12 scenes, B2 only 3. B1's failures are mostly one thing left unused ([2] the teapot, [8] the alarm clock). B2 often leaves two of its three things unused ([1] chair and door, [8] broom and wristwatch, [9] house plant and alarm clock, [10] sink and door), and its sentences break more: [1] "read my book and pill ... I'd wear my old pillow to sleep until my morning pillow", [8] "I's hardly home". Even B2's passing scene [5] picks the alarm clock twice and says "so her breakfast long before I wake".

B2f6 passes 8 of these 12, more than B2 (3) and B1 (7), but a pass is only the program's check, and several of its passing lines are not good sentences: [1] "nap on my chair by the chair I'm pouring pints", [10] "in my sunnoon's sun", [12] "A small vase of white hair from a bathtub", [5] "hang up my pillow ... wipe off my pillow". It still picks things it does not use ([2] chair, [3] car and bus, [7] pillow, [11] house plant) and picks the door twice in [7].

## 6. Every kept line, side by side: the new validation set

The same 48 kinds of prompt, from the 17 new held-out sales.

### Openings

In [11]:
show("new", "open")

[1] Chloe (sale 2204)
    (nothing yet)
  sale: Hi! I'm paying rent on a shared flat out of a part time café job, so I'm hoping for a kitten that won't break me. | $55
  B1:    I'm a student on a tight budget, so I'm looking for a good deal. | $60
        check: {"readable": true, "offer": 60, "spoken_ok": true}
  B2:    I'm the one on a budget, so I know what a cat needs to do, but I'd love that bills. | $55
        check: {"readable": true, "offer": 55, "spoken_ok": true}
  B2f6:  I'm a second year chemistry student, so every dollar I spend on a cat is already spoken for. | $55
        check: {"readable": true, "offer": 55, "spoken_ok": true}

[2] Jonah (sale 2204)
    (nothing yet)
  sale: Four lanes of traffic run right past my front step, day and night. | $72
  B1:    Hi. I'm on a busy road. | $55
        check: {"readable": true, "offer": 55, "spoken_ok": true}
  B2:    Lorries rumble past my front step on Sorrel Parade from dawn to midnight, so whoever comes home with me has to 

Here the picture turns around. B2's openings sound like the written sales: concrete and tied to the card (Jonah: "Lorries rumble past my front step ... whoever comes home with me has to stay in"; Leo: "the house ... has been far too quiet since"), while B1's are short and generic ("Hi. I'm on a busy road.", "Student here, small budget and small budget."). B2's offers are much closer to the sale: about $5 away on average against about $17 for B1, which opens too low (Owen $70 where the breeder in the sale offers $110). But B2's longer sentences break more often: [6] "attoo is two", [3] "full of sunny grease and brass", [1] "I'd love that bills", [7] "I can't help a single room with my neighbour's books".

B2f6's openings are concrete in the same way as B2's ("I'm a second year chemistry student, so every dollar I spend on a cat is already spoken for"; "Fitz slept beside my workshop every evening") and their offers are close to the sale (about $6 off). Some break or wander: [9] the hiker Ben says "My back garden on Fenwick Parade is on theaming table", [7] Felix, whose card is a small flat and a new couch, says "I drive the night bus", and [3] Hana "stand[s] guard over half my living room all day".

### Questions

In [12]:
show("new", "ask")

[1] Owen (sale 2204)
    Your opening ($110): My cattery has produced four grand champions, and every kitten I add has to strengthen a bloodline.
  sale: Is she a pedigree, and does she come with any registration papers?
  B1:    Is the cat the pedigree cat and does it have any specific breed traits that would make her suitable for breeding?
        check: {"readable": true, "new_question": true, "new_in_meaning": true}
  B2:    Before anything else, is she still intact, and is she still entire for breeding?
        check: {"readable": true, "new_question": true, "new_in_meaning": true}
  B2f6:  Before anything else, is he a pedigreed breed with papers, and is he still intact?
        check: {"readable": true, "new_question": true, "new_in_meaning": true}

[2] Jonah (sale 2204)
    Owen asked: Is she a pedigree, and does she come with any registration papers?
    Seller: She's a two month old black domestic shorthair, and that coat shines like wet ink. No papers that I know of, but she

B2's questions follow the asker's card more often: the breeder asks whether she is still entire [1], the parent asks about small children [4], the allergic buyer about fur [8], the hiker about a harness [12]. B1 has several broken questions: [4] "Is the kitten shots being a monthly ...", [11] "a purebred Bush", and [3] turns a question into a walk-away ("so I'm out. | out"). B2 still slips: [6] asks about going outdoors, which was already answered, [9] the buyer with a new couch asks about other cats instead of scratching, and [11] calls the cat "he".

B2f6's questions also follow the cards (the breeder asks for papers and whether he is intact, the parent about small children, the hiker about a harness), but [3] Chloe, who asked nothing about furniture before, asks whether the kitten "could scratch up my café?", [8] Aisha asks whether the cat has been "tested for Fel d 1", [9] Felix, with the new couch, asks about shedding instead of scratching, and [1] and [11] call the cat "he".

### Reactions

In [13]:
show("new", "react")

[1] Chloe (sale 2204)
    Your opening ($55): Hi! I'm paying rent on a shared flat out of a part time café job, so I'm hoping for a kitten that won't break me.
    Owen asked: Is she a pedigree, and does she come with any registration papers?
    Seller: She's a two month old black domestic shorthair, and that coat shines like wet ink. No papers that I know of, but she carries herself like a champion already.
  sale: 0 | No papers keeps her at a market price, which is the only kind of price my café wages can manage. Holding at $55. | $55
  B1:    0 | Papers don't matter to me. $55. | $55
        check: {"readable": true, "rules_ok": true, "same_direction": true, "withdrew": false, "teacher_withdrew": false, "offer_error": 0, "said_equals_offer": true, "reasonable": true}
  B2:    0 | Papers never cost me a cent, and a breed name won't make my rent any cheaper to keep. Staying at $55. | $55
        check: {"readable": true, "rules_ok": true, "same_direction": true, "withdrew": false, "t

B2's reasons tie the answer to the buyer's own life more often, as the new sales teach: Chloe [1] "Papers never cost me a cent, and a breed name won't make my rent any cheaper" against B1's "Papers don't matter to me. $55." Both models miss the same three raises, where the written buyers saw a plus in a two-month-old kitten ([4] learning odd hours, [9] riding in a pack, [10] sixteen more years). Both get [12] wrong: the cat has been indoor-outdoor, which is bad news on a busy road, yet both raise, and B2 even reads it as "she stays safe inside". B2 also invents or contradicts facts: [6] "one month old" (the seller said two) and [2] "the traffic was never my worry" from a buyer whose card is the traffic. [11] fails the rules.

B2f6's reasons are as personal as B2's and in one place better: [2] Jonah keeps his worry about the road ("Papers don't keep her off the road outside my house") where B2 said the traffic was never his worry. It misses the same three raises for a young kitten ([4], [9], [10]) and makes the same wrong raise at [12] ("An indoor cat who stays safely inside is the best news"). It invents a fact at [4] ("Every cat in my home has papers") and garbles [5] ("my pettery papers, and I'm notang") and [7] ("keeps a cat out of the clay"); [6] fails the rules.

### Scenes

In [14]:
show("new", "imagine")

[1] Chloe (sale 2204)
    Hana asked: Does she have any funny little habits I should know about before taking her home?
    Seller: Just one, and it's adorable: she'll only drink from a running tap. Honestly, watching her lap at the stream is like a little photo shoot every time.
    You ($41 -> $35): A tap left running for her drinks would push up the water bill I split with three flatmates. Down to $35.
  sale: laptop, book, coffee cup | I'd study at my laptop with a secondhand book open while she naps nearby, and my coffee cup keeps me awake through her noisy evenings.
  B1:    book, backpack, calculator | She'd nap on my book while I study with my book, I'd tally her spay on my calculator, and carry it home in my backpack.
        check: {"readable": true, "rules_ok": true}
  B2:    broom, spreadsheet, book | I'd clean her with my broom every morning after a café shift, then spreadsheet her vet costs on my spreadsheet.
        check: {"readable": true, "rules_ok": false} — the line

On the new set B1 passes 8 of the 12 kept scenes and B2 only 1. B2's scenes often have nothing to do with the things it picked: [2] scissors, door and flashlight become "cut paper cloth ... so she can sit on the road" and a microphone; [5] map, shoe and school bus become a sentence about a salsa shoe; [12] it picks "bike", which is not on the drawable list. Words break apart: "theuss", "the walnut pillow I built", "I'd hang my bike in the hall after every bike". [10] even ends without a full stop, which may be the length limit cutting it off. The written scenes in this set are long and personal (Elena's [5] is 39 words), and B2 seems to imitate that length without being able to keep track of the three things.

B2f6 passes 1 of these 12, like B2, and fails in a way the others do not here: three lists are not three things (‹hat›chen‹sink›, ‹scissors› weekends‹pillow›, bike‹sock›‹calendar›). It picks the same thing twice ([4] the alarm clock, [7] the washing machine, [8] the backpack), uses things it did not pick ([3] a fencing sword, [7] a sink), and gives Hana's black kitten "orange stripes". Its lines also repeat themselves ("leave my old cat a pillow by the pillow, and let her watch me knead my pillow after the pillow").

## 7. Why are B2's scenes worse?

Three kinds of mistake show up in the scenes above: **(1)** a thing is picked but never used in the line, **(2)** the
line names a thing that was not picked or is not on the drawable list, **(3)** the sentence itself breaks. The first
two are counted by the program check; the third has to be read. B2f6 adds a fourth: a list that is not three
things. First the counts over the 24 kept scenes of each model:

In [15]:
def kind(row):
    problem = row["checks"].split(" — ")[-1] if " — " in row["checks"] else ""
    things = scene_things(row["model"])
    return {"(1) picked, not used": "never names" in problem,
            "(2) not picked / not drawable": "without picking" in problem or "not on the list" in problem,
            "not three things": "exactly 3" in problem,
            "same thing picked twice": len(things) != len(set(things)),
            "passes": not problem}

for label, col in zip(MODELS, range(3)):
    total = Counter()
    for which in ("old", "new"):
        for p in pairs(which):
            if p[0]["task"] == "imagine":
                total.update(k for k, v in kind(p[col]).items() if v)
    print(label, dict(total))

B1 {'(2) not picked / not drawable': 2, '(1) picked, not used': 7, 'passes': 15, 'same thing picked twice': 2}
B2 {'(1) picked, not used': 15, '(2) not picked / not drawable': 5, 'passes': 4, 'same thing picked twice': 4}
B2f6 {'passes': 9, '(1) picked, not used': 11, 'same thing picked twice': 4, 'not three things': 3, '(2) not picked / not drawable': 1}


Over the 24 kept scenes per model, B1 passes 15, B2 4 and B2f6 9. B2 leaves picked things unused in 15 scenes, B2f6 in 11 and B1 in 7; B2 names unpicked or undrawable things in 5 (B2f6 1, B1 2); each of B2 and B2f6 picks the same thing twice in 4 scenes (B1 2); and B2f6 writes 3 lists that are not three things. The third kind, broken sentences, is not counted by any rule; the lines above show it in B2 and B2f6 far more often than in B1.

One possible cause is the length limit: in the game and in the evaluation a scene may use at most `MAX_NEW["imagine"]`
tokens, and a line cut short would leave its last thing unused. How long are the written scenes?

In [16]:
import sentencepiece as spm
sp = spm.SentencePieceProcessor(model_file=str(ML / "checkpoints/tokenizer/sp8192.model"))
print("limit for a scene:", MAX_NEW["imagine"], "tokens")
for label, path in (("B1 training", "data/processed/buyers_b1/train.jsonl"),
                    ("B2 training", "data/processed/buyers_b2/train.jsonl"),
                    ("old set", "runs/eval_sets_b2/val_old_format5.jsonl"),
                    ("new set", "runs/eval_sets_b2/val_new_format5.jsonl")):
    n = sorted(len(sp.encode(json.loads(line)["target"])) for line in open(ML / path)
               if json.loads(line)["task"] == "imagine")
    over = sum(x > MAX_NEW["imagine"] for x in n)
    print(f"{label:12} {len(n):5} scenes, median {n[len(n) // 2]} tokens, longest {n[-1]}, "
          f"over the limit {over} ({over / len(n):.0%})")

limit for a scene: 64 tokens
B1 training    762 scenes, median 40 tokens, longest 63, over the limit 0 (0%)


B2 training   1987 scenes, median 50 tokens, longest 85, over the limit 111 (6%)


old set         89 scenes, median 41 tokens, longest 58, over the limit 0 (0%)


new set        152 scenes, median 56 tokens, longest 76, over the limit 17 (11%)


B1 never saw a scene longer than the 64-token limit; 6% of B2's training scenes and 11% of the new set's are longer. So some of B2's scenes are cut before they finish, which leaves the last thing unused (the cut-off line [10] of the new set may be one of them). But most of B2's failed scenes above are finished sentences that simply do not use their things, so the limit explains only a small part.

Another difference between the old and the new sales is which things their scenes use. How many different things
does each group of training scenes pick, and how often is a thing picked that is used fewer than five times?

In [17]:
def things(path, keep):
    count, scenes = Counter(), 0
    for line in open(ML / path):
        r = json.loads(line)
        if r["task"] == "imagine" and keep(r["seed"]):
            scenes += 1
            count.update(t.strip() for t in r["target"].split("|")[0].split(","))
    return count, scenes

for label, path, keep in (("B1 training", "data/processed/buyers_b1/train.jsonl", lambda s: True),
                          ("B2 training, old sales", "data/processed/buyers_b2/train.jsonl", lambda s: s < 2200),
                          ("B2 training, new sales", "data/processed/buyers_b2/train.jsonl", lambda s: s >= 2200)):
    count, scenes = things(path, keep)
    rare = sum(n for n in count.values() if n < 5) / sum(count.values())
    print(f"{label:24} {scenes:5} scenes, {len(count):3} different things, rare picks {rare:.0%}, "
          f"most used {count.most_common(4)}")

B1 training                762 scenes,  96 different things, rare picks 2%, most used [('pillow', 90), ('door', 89), ('picture frame', 72), ('alarm clock', 61)]

B2 training, old sales     762 scenes,  96 different things, rare picks 2%, most used [('pillow', 90), ('door', 89), ('picture frame', 72), ('alarm clock', 61)]
B2 training, new sales    1225 scenes, 231 different things, rare picks 5%, most used [('pencil', 75), ('dog', 72), ('scissors', 69), ('book', 68)]


The old sales picked from 96 different things, again and again (pillow 90 times, door 89). The new sales pick from 231 different things, and the most used one, pencil, appears only 75 times in 1,225 scenes. With more than twice as many things, each one is seen far less often, so the model has fewer examples of how to use any one of them. Together with the longer scenes, this is a likely reason why B2's scenes got worse; section 8 tests how much of it holds.

## 8. Experiments: what makes the scenes fail?

The numbers above suggest causes but do not test them. These runs do (`tools/probe_imagine.py`, on the CPU, every
imagine prompt of both sets, 89 + 152 = 241 scenes):

- **b1_64, b2_64**: each model picks its own things and writes the line, with the usual limit of 64 tokens.
- **b2_128**: the same, but B2 may write up to 128 tokens. If cut-off lines were the problem, this would fix it.
- **b1_forced, b2_forced**: the scene starts with the three things the written sale picked, and the model only
  writes the line. This separates *choosing* things from *writing* about them.

The CPU draws different random numbers from the Mac's GPU, so these are not the same samples as section 3; the runs
are compared with each other.

In [18]:
from sellanything_ml.buyers.stories import _named

PROBE = ML / "runs/sft_b2/imagine_probe"
probe = {name: [json.loads(line) for line in open(PROBE / f"{name}.jsonl")]
         for name in ("b1_64", "b2_64", "b2_128", "b1_forced", "b2_forced")}


def kind(r):
    p = r.get("problem") or ""
    if r.get("rules_ok"):
        return "passes"
    if not r.get("readable"):
        return "unreadable"
    if "never names" in p:
        return "picked, not used"
    if "without picking" in p:
        return "not picked"
    if "not on the list" in p or "cannot be drawn" in p:
        return "not drawable"
    return "other"


for name, rows in probe.items():
    rate = {s: sum(bool(r.get("rules_ok")) for r in rows if r["set"] == s) / sum(r["set"] == s for r in rows)
            for s in ("old", "new")}
    print(f"{name:10} old {rate['old']:4.0%}  new {rate['new']:4.0%}  cut off {sum(r['hit_limit'] for r in rows):3}  "
          f"{dict(Counter(kind(r) for r in rows))}")

b1_64      old  70%  new  68%  cut off   0  {'passes': 165, 'picked, not used': 65, 'not drawable': 2, 'not picked': 8, 'unreadable': 1}
b2_64      old  42%  new  20%  cut off  15  {'passes': 67, 'picked, not used': 157, 'not picked': 7, 'other': 4, 'not drawable': 6}
b2_128     old  42%  new  20%  cut off   0  {'passes': 68, 'picked, not used': 156, 'not picked': 7, 'other': 4, 'not drawable': 6}
b1_forced  old  44%  new   1%  cut off   0  {'passes': 41, 'picked, not used': 194, 'not picked': 5, 'unreadable': 1}
b2_forced  old  28%  new   9%  cut off   0  {'picked, not used': 200, 'passes': 39, 'not picked': 2}


**The length limit is not the cause.** With 128 tokens instead of 64, B2 passes 28.2% instead of 27.8%: of the 15 lines that were cut off at 64, only one more passes when it may finish.

**Choosing is where B1's scores come from.** Given the written sale's three things, B1 passes 44% of the old set and only **1%** of the new set. Its 68% on the new set (choosing its own things) comes from picking things it knows how to write about, not from being able to write about the things the new sales use. B2, given the same things, passes 28% of the old set and 9% of the new set: a little better than B1 on the new things, but far from usable.

Almost every failure in every run is the same kind: a picked thing that the line never uses.

Then, for every thing a model put in its list, was it used in the line? Grouped by how many training scenes of that
model picked the same thing, and by its place in the list:

In [19]:
seen = {}                                 # how many training scenes picked each thing
for label, path in (("b1", "data/processed/buyers_b1/train.jsonl"), ("b2", "data/processed/buyers_b2/train.jsonl")):
    seen[label] = Counter()
    for line in open(ML / path):
        r = json.loads(line)
        if r["task"] == "imagine":
            seen[label].update(t.strip() for t in r["target"].split("|")[0].split(","))

BUCKETS = ("never", "1-9", "10-29", "30+")
def bucket(n):
    return "never" if n == 0 else "1-9" if n < 10 else "10-29" if n < 30 else "30+"

for name, rows in probe.items():
    used, picked, unused_at = Counter(), Counter(), [0, 0, 0]
    for r in rows:
        if "|" not in r["model"]:
            continue
        things, line = r["model"].split("|", 1)
        things = [t.strip() for t in things.split(",")]
        if len(things) != 3:
            continue
        for k, t in enumerate(things):
            b = bucket(seen[name[:2]][t])
            picked[b] += 1
            if _named(t, line):
                used[b] += 1
            else:
                unused_at[k] += 1
    rates = "  ".join(f"{b}: {used[b]}/{picked[b]} {used[b] / picked[b]:.0%}" for b in BUCKETS if picked[b])
    print(f"{name:10} {rates}   unused 1st/2nd/3rd thing {unused_at}")

b1_64      never: 1/2 50%  1-9: 17/19 89%  10-29: 49/56 88%  30+: 570/643 89%   unused 1st/2nd/3rd thing [26, 25, 32]
b2_64      never: 2/6 33%  1-9: 6/8 75%  10-29: 60/94 64%  30+: 434/603 72%   unused 1st/2nd/3rd thing [76, 70, 63]
b2_128     never: 2/6 33%  1-9: 6/8 75%  10-29: 61/94 65%  30+: 434/603 72%   unused 1st/2nd/3rd thing [76, 69, 63]
b1_forced  never: 4/177 2%  1-9: 36/126 29%  10-29: 34/68 50%  30+: 271/352 77%   unused 1st/2nd/3rd thing [124, 131, 123]
b2_forced  never: 0/3 0%  1-9: 12/51 24%  10-29: 89/149 60%  30+: 344/520 66%   unused 1st/2nd/3rd thing [100, 93, 85]


**The rarer the thing, the less often it gets used.** When the things are given, B1 uses a thing it never saw in a training scene 2% of the time, one it saw 1–9 times 29%, and one it saw 30 times or more 77%; B2 uses 24% of the things seen 1–9 times and 66% of those seen 30 times or more. The new sales picked from 231 different things (section 7), many of them seen only a few times, so this explains a large part of the drop on the new set.

**But rarity is not the whole story.** When B2 picks its own things they are mostly common ones (603 of its 711 picks were seen 30 times or more), and it still uses only 72% of those, against B1's 89%. The unused thing is spread evenly over the first, second and third place, so it is not a line running out of room. Something in how B2 learned to write scenes also got worse; the longer, more personal scene lines of the new sales (29 words against 21) are a likely reason, but that is not tested here.

Do the models read their list at all? A teacher-forced test (`tools/probe_scene_list.py`): give the model the
prompt, the written scene's list and the line up to the word that names a thing, and take the probability of that
word. Then swap the thing in the list for another drawable thing (umbrella, basket, ...) and score the same word
again. A model that writes from its list gives the word a much lower probability once the thing is off the list.

In [20]:
LIST = ML / "runs/sft_b2/scene_list"
swap = {m: [json.loads(line) for line in open(LIST / f"{m}.jsonl")] for m in ("b1", "b2")}
for m, rows in swap.items():
    groups = {}
    for r in rows:
        groups.setdefault((r["set"], bucket(seen[m][r["thing"]])), []).append(r)
    print(f"{m}: {len(rows)} things, mean drop {sum(r['drop'] for r in rows) / len(rows):.2f}")
    for (which, b), g in sorted(groups.items()):
        if len(g) < 10:
            continue
        listed = sorted(math.exp(r["logp_listed"]) for r in g)[len(g) // 2]
        swapped = sorted(math.exp(r["logp_swapped"]) for r in g)[len(g) // 2]
        print(f"   {which} set, seen {b:5} n={len(g):3}  p(word | on the list) {listed:.3f}   "
              f"p(word | swapped out) {swapped:.4f}")

b1: 723 things, mean drop 3.20
   new set, seen 1-9   n=104  p(word | on the list) 0.016   p(word | swapped out) 0.0001
   new set, seen 10-29 n= 38  p(word | on the list) 0.017   p(word | swapped out) 0.0009
   new set, seen 30+   n=138  p(word | on the list) 0.395   p(word | swapped out) 0.0066
   new set, seen never n=176  p(word | on the list) 0.000   p(word | swapped out) 0.0000
   old set, seen 1-9   n= 22  p(word | on the list) 0.626   p(word | swapped out) 0.0028
   old set, seen 10-29 n= 30  p(word | on the list) 0.987   p(word | swapped out) 0.4852
   old set, seen 30+   n=214  p(word | on the list) 0.995   p(word | swapped out) 0.5727
b2: 723 things, mean drop 3.76
   new set, seen 1-9   n= 48  p(word | on the list) 0.183   p(word | swapped out) 0.0001
   new set, seen 10-29 n=126  p(word | on the list) 0.442   p(word | swapped out) 0.0019
   new set, seen 30+   n=279  p(word | on the list) 0.742   p(word | swapped out) 0.0095
   old set, seen 10-29 n= 23  p(word | on the li

**B1 writes common things from habit; B2 writes from its list.** On the old set, with a common thing on the list, B1 gives its word a probability of 0.995, and still **0.573** after the thing is swapped out of the list: B1 would write "pillow" or "door" whether or not it had picked them. B2 gives the word 0.976 when it is listed and only **0.052** when it is not, so B2's line follows its list. Together with notebook 4 (train loss 0.41 against validation loss 0.906, and scene lines copied word for word from the training data), this says B1's good scene score comes largely from repeating the few things it saw most, not from writing about what it chose.

What this test does **not** show is whether either model learned which buyers prefer which things: it tests the step from the list to the line, not from the buyer to the list. B1's picks are crowded into its 20 most common things (51% of its training picks), which fits "not learned", but that is not proof.

**B2's trouble is the rare things, and its own drifting lines.** With a thing on the list that it saw only 1–9 times, B2 gives the word a median probability of 0.18 on the new set (0.44 for 10–29 times, 0.74 for 30 or more). In this test the model is put exactly where the word should come; writing on its own, B2's longer lines often wander off before they get there, which is why it also misses common things.

Does it matter how many tokens a thing's name takes? The tokenizer cuts some names into two or more pieces
("teapot" may be one token, "stethoscope" several). Within the same frequency group:

In [21]:
import sentencepiece as spm
sp = spm.SentencePieceProcessor(model_file=str(ML / "checkpoints/tokenizer/sp8192.model"))
for name in ("b1_forced", "b2_forced", "b2_64"):
    used, total = Counter(), Counter()
    for r in probe[name]:
        if "|" not in r["model"]:
            continue
        things, line = r["model"].split("|", 1)
        things = [t.strip() for t in things.split(",")]
        if len(things) != 3:
            continue
        for t in things:
            key = ("30+" if seen[name[:2]][t] >= 30 else "<30",
                   "1 token" if len(sp.encode(" " + t.split()[-1])) == 1 else "2+ tokens")
            total[key] += 1
            used[key] += _named(t, line)
    print(name, "  ".join(f"seen {f}, {n}: {used[(f, n)]}/{total[(f, n)]} {used[(f, n)] / total[(f, n)]:.0%}"
                         for f, n in sorted(total)))

b1_forced seen 30+, 1 token: 184/235 78%  seen 30+, 2+ tokens: 87/117 74%  seen <30, 1 token: 41/148 28%  seen <30, 2+ tokens: 33/223 15%
b2_forced seen 30+, 1 token: 218/313 70%  seen 30+, 2+ tokens: 126/207 61%  seen <30, 1 token: 39/70 56%  seen <30, 2+ tokens: 62/133 47%
b2_64 seen 30+, 1 token: 212/268 79%  seen 30+, 2+ tokens: 222/335 66%  seen <30, 1 token: 20/34 59%  seen <30, 2+ tokens: 48/74 65%


Names cut into several tokens are used a little less often even among equally common things (B2 given the things: 70% for one-token names against 61% for longer ones, seen 30 times or more; B1: 28% against 15% for the rarer ones). So one token per thing could help a little, but how often a thing was seen matters much more. Section 10 tests one token per thing directly.

## 9. What this comparison says

1. **Reactions, the core of the game, are better in B2 than B1 on the new style of sale, and B2f6 is about as good as B2.** On the 17 new held-out sales B2 and B2f6 move the offer the way the written buyer did more often than B1 (74.2% and 72.8% against 67.4%), land closer to the written offer ($2.93 and $3.38 against $4.62), and walk away a sensible number of times (14 and 22 against the sale's 18; B1 39). B2f6 passes the rules a little more often (93.1%) but says a sign that does not match its own offer more often.
2. **On the old style of sale no model is clearly better.** All three misread the seller in the same places ("fully vaccinated" heard as "no vaccinations"); B2 lets the breeder lower instead of walking away from an FIV-positive cat, B2f6 gets that one right but raises for a reason that ignores FIV.
3. **B1's scenes look better than they are.** B1 writes its most common things whether or not they are on its list (0.57 probability after swapping them out), so its scene score comes largely from repeating the few things it saw most. B2 writes from its list (0.05 after swapping); B2f6 mostly does too (0.11).
4. **B2's scenes fail on rare things and drifting lines.** The 64-token limit is not the cause. The 344 drawable things are spread thinly over 1,987 training scenes (109 never picked, 97 picked 1–9 times), and the less often a thing was seen, the less likely its word; B2's longer lines also wander before they reach the word.
5. **Format 6 (one token per thing, persona and animal; section 10) did not fix the scenes.** Its scenes pass more often on the old set in both draws (50.6% against 38.2% in the evaluation, 49% against 42% on the CPU) but not on the new set (14.5% against 11.9% in the evaluation, 14% against 20% on the CPU; 27.4% against 27.8% overall). It made the rare things much worse: with a thing seen 1–9 times on its list, it gives that word a median probability of 0.012 against B2's 0.183, because B2 can copy a written name from its list and B2f6 has to learn each token's word from too few scenes. A failure also grew: 13% of its lists are not three thing tokens (B2 2%), because at a list place it puts 23% of the probability outside the thing tokens on the new set. Things seen 10–29 times did get easier (0.579 against 0.442).
6. **The persona token did not tie the buyer to the list more strongly.** Swapping the persona lowers the written list less in B2f6 (0.74 and 0.14 nats per thing on the old and new sets) than in B2 (0.92 and 0.25).
7. **Hearing the seller did not get worse.** Read line by line, B2f6 misreads 2 of 140 reactions to "fully vaccinated" and 1 of 245 ages (B2 8 and 1, B1 6 and 6); the counts are small and clustered, so this is no evidence of a gain either.
8. **What to change next, all training and data:** (a) in the sales still to be written, give the rarely seen things many more scenes (about 30 each would need some 10,000 picks, roughly 3,400 scenes); this is what both formats lack, and format 6 needs it most. (b) Keep scene lines short, about one clause per thing. (e) More weight on the scene examples (8% of all examples); the training loss of both B2 and B2f6 ends near 0.6 against a validation loss above 1.3, so more weight on the same few scenes would likely be memorised. Format 6 is worth trying again only once each thing has enough scenes.
9. **What this comparison cannot show:** every line here starts from a written story, so it does not show how the models do in a live sale, where their own earlier mistakes would carry forward.

## 10. Format 6: a token for every thing, every persona and the animal

Section 9 proposed five changes for the scenes. B2f6 tests two of them, **(c)** one token per drawable thing and
**(d)** the buyer's persona and the kind of animal as tokens, and nothing else: the same 250 sales, the same split,
the same recipe as B2. The other three, (a) more scenes for rare things, (b) shorter scene lines and (e) more weight on
scene examples, are not in it. Each new token starts as the mean of the embeddings of its words ("alarm clock" →
mean of "alarm" and "clock") and is then learned during fine-tuning.

In [22]:
from sellanything_ml.buyers.format import MEANINGS, N_BASE_EXTRA, THING_TOKENS
from sellanything_ml.llm.tokenizer import TOKENIZER_DIR, Tokenizer

tok = Tokenizer(TOKENIZER_DIR / "sp8192.model")         # knows the added tokens (plain sentencepiece does not)

print(f"new tokens: {len(ANIMAL_TOKENS)} animals, {len(BUYER_TOKENS)} personas, {len(THING_TOKENS)} things "
      f"= {len(MEANINGS)}; with the {N_BASE_EXTRA} added before, {N_BASE_EXTRA + len(MEANINGS)} added to the 8,192 words")
rows = [json.loads(line) for line in open(F6 / "metrics.jsonl")]
print("B2f6 validation loss:", " ".join(f"{r['step']}:{r['val_loss']:.3f}" for r in rows))
for name, path in (("B2", "data/processed/buyers_b2/train.jsonl"), ("B2f6", "data/processed/buyers_b2_f6/train.jsonl")):
    scenes = [json.loads(line)["target"] for line in open(ML / path) if json.loads(line)["task"] == "imagine"]
    lists = [len(tok.encode(t.split(" | ")[0])) for t in scenes]
    whole = sorted(len(tok.encode(t)) for t in scenes)
    print(f"{name}: a written scene's list is {sum(lists) / len(lists):.1f} tokens on average, the whole scene "
          f"{whole[len(whole) // 2]} (median); longer than the {MAX_NEW['imagine']}-token limit: "
          f"{sum(x > MAX_NEW['imagine'] for x in whole)} of {len(whole)}")

new tokens: 2 animals, 17 personas, 344 things = 363; with the 14 added before, 377 added to the 8,192 words
B2f6 validation loss: 500:1.710 1000:1.570 1500:1.475 2000:1.444 2500:1.394 3000:1.334 3500:1.363 4000:1.357 4500:1.345 4844:1.342


B2: a written scene's list is 8.7 tokens on average, the whole scene 50 (median); longer than the 64-token limit: 111 of 1987


B2f6: a written scene's list is 3.0 tokens on average, the whole scene 45 (median); longer than the 64-token limit: 9 of 1987


Format 6 adds 363 tokens (2 animals, 17 personas, 344 drawable things), so with the 14 tokens added before, the vocabulary grows from 8,192 to 8,569. The validation loss behaves like B2's: lowest at step 3000, a little higher after. Because a thing is one token, a written list shrinks from 8.7 tokens to 3.0, the median scene from 50 to 45 tokens, and only 9 written scenes are longer than the 64-token limit instead of 111, so the length limit, already a small factor (section 8), almost disappears.

### 10.1 The scene experiments again, with B2f6

The same CPU runs as section 8 (`tools/probe_imagine.py --sets runs/eval_sets_b2f6`): B2f6 picks its own things
(**b2f6_64**), and B2f6 is given the written sale's three things as tokens (**b2f6_forced**). A list that is not
three thing tokens fails as "not three things".

In [23]:
probe["b2f6_64"], probe["b2f6_forced"] = ([json.loads(line) for line in open(F6 / f"imagine_probe/{name}.jsonl")]
                                         for name in ("b2f6_64", "b2f6_forced"))


def kind6(r):
    return "not three things" if "exactly 3" in (r.get("problem") or "") else kind(r)


for name in ("b1_64", "b2_64", "b2f6_64", "b1_forced", "b2_forced", "b2f6_forced"):
    rows = probe[name]
    rate = {s: sum(bool(r.get("rules_ok")) for r in rows if r["set"] == s) / sum(r["set"] == s for r in rows)
            for s in ("old", "new")}
    twice = sum(len(scene_things(r["model"])) != len(set(scene_things(r["model"]))) for r in rows)
    print(f"{name:12} old {rate['old']:4.0%}  new {rate['new']:4.0%}  cut off {sum(r['hit_limit'] for r in rows):3}  "
          f"same thing twice {twice:3}  {dict(Counter(kind6(r) for r in rows))}")

b1_64        old  70%  new  68%  cut off   0  same thing twice   8  {'passes': 165, 'picked, not used': 65, 'not drawable': 2, 'not picked': 8, 'unreadable': 1}
b2_64        old  42%  new  20%  cut off  15  same thing twice  31  {'passes': 67, 'picked, not used': 157, 'not picked': 7, 'not three things': 4, 'not drawable': 6}
b2f6_64      old  49%  new  14%  cut off   3  same thing twice  39  {'picked, not used': 134, 'passes': 66, 'not picked': 9, 'not three things': 32}
b1_forced    old  44%  new   1%  cut off   0  same thing twice   0  {'passes': 41, 'picked, not used': 194, 'not picked': 5, 'unreadable': 1}
b2_forced    old  28%  new   9%  cut off   0  same thing twice   0  {'picked, not used': 200, 'passes': 39, 'not picked': 2}
b2f6_forced  old  31%  new   5%  cut off   0  same thing twice   0  {'picked, not used': 198, 'passes': 36, 'not picked': 7}


**No better overall.** Choosing its own things, B2f6 passes 49% of the old set (B2 42%, B1 70%) and 14% of the new set (B2 20%, B1 68%): 27.4% of all 241 scenes against B2's 27.8%. On the old set B2f6 is ahead of B2 in both draws (50.6% against 38.2% in section 3, 49% against 42% here); on the new set, where the rare things are, it is ahead in section 3 (14.5% against 11.9%) and behind here (14% against 20%), so on the new set no gain survives a second draw.

Fewer lines are cut off (3 against 15), as the shorter lists predict, but another failure grows in their place: **32 lists (13%) are not three things** (B2 4, B1 none), and the same thing is picked twice more often (39 times against 31). Given the written sale's three things, B2f6 passes 31% of the old set and 5% of the new set (B2 28% and 9%, B1 44% and 1%), so writing about given things did not get better either. Almost every other failure is still a picked thing that the line never uses.

What is in the lists that are not three things? The words written between or instead of the thing tokens:

In [24]:
from sellanything_ml.buyers.stories import DRAWABLE_THINGS

broken = [r for r in probe["b2f6_64"] if "exactly 3" in (r.get("problem") or "")]
words = Counter(w for r in broken for w in THING.sub(" ", r["model"].split(" | ")[0]).split())
print(len(broken), "broken lists; the words in them:", dict(words.most_common(12)))
print("words that are drawable things:", sum(n for w, n in words.items() if w in DRAWABLE_THINGS), "of", sum(words.values()))
for r in broken[:8]:
    print("   ", THING.sub(r"‹\1›", r["model"])[:110])

32 broken lists; the words in them: {'weekends': 6, 'kit': 3, 'wine': 3, 'ill': 2, 'cup': 1, 'Her': 1, 'coat': 1, 'way': 1, 'bike': 1, 'ur': 1, '+': 1, 'bed': 1}
words that are drawable things: 8 of 36
    ‹laptop›‹pencil› cup | She'd nap on my laptop in the laptop while I fill a cup of water, and I'd cook her a li
    Her coat | Her coat would look stunning next to a clean vase of flowers, and I'd brush her with a soft toothbr
    ‹pillow› kit‹pillow› | I'd give her her own pillow in my small flat, and she'd sleep on a pillow in my small f
    ‹hat›‹door›way | I'd hang my old sweater up high where her can stay, and she'd watch me fit a door in my hat w
    bike‹pencil›‹envelope› | I'd check my wristwatch to feed her before every bike and her scrap, keep my backpack
    ‹bicycle›‹hammer› weekends | I'd hang my bicycle in the hall so she can sniff it, hang my bicycle up high, and
    ur‹washing machine›‹bathtub› | I'd wash my hands at the sink after brushing her, run my washing machine 

The broken lists are not drawable names written as words instead of tokens: only 8 of the 36 words are drawable things (cup, flower, broom, door, candle, bed, garden, bird). The rest are pieces of words ("kit", "ill", "li", "ur", "es") and other words; " weekends" alone appears 6 times, and none of those six buyers is the hiker whose persona contains that word. At the place of a thing the model can write any of its 8,569 tokens, and it has not learned to keep to the 344 thing tokens. The next cell measures how much probability leaves them.

Where does the probability go at a list place? A teacher-forced test (`tools/probe_list_tokens.py`): at each of
the three places of every written list, the probability (at the game's temperature 0.7) of all 344 thing tokens
together, and of the thing the sale wrote:

In [25]:
slot = json.load(open(F6 / "list_tokens.json"))
for which in ("old", "new"):
    print(which, slot[which])
print("most likely other pieces:", slot["most likely other pieces"])

old {'mean p(any thing token)': 0.9729, 'places under 0.9': 11, 'median p(written thing)': 0.3492, 'by place': [0.9889, 0.9772, 0.9524]}
new {'mean p(any thing token)': 0.7718, 'places under 0.9': 420, 'median p(written thing)': 0.0042, 'by place': [0.8113, 0.7565, 0.7478]}
most likely other pieces: {' weekends': 188, 's': 90, ' kit': 86, ' furn': 67, 'ed': 66, 'es': 60, ' broom': 60, ' bed': 48, ' door': 48, 'ing': 43, ' |': 42, 'le': 40, 'ill': 39, ' hair': 33, '+': 32}


**The model has not learned that a list place takes a thing token.** On the old set the 344 thing tokens together get 97% of the probability at a list place; on the new set only 77%, and less than 90% at 420 of its 456 places. The thing the sale wrote gets a median of 0.35 on the old set but only 0.004 on the new set, where the sales pick from 231 different things, many of them rare. Where the model is unsure which thing comes next, the probability spreads to ordinary pieces: " weekends" is among the three most likely of them at 188 of the 723 places, then "s", " kit", " furn", "ed", "es", drawable words written as text (" broom", " bed", " door") and " |", which would end the list early. Sampling with top-k 50 cuts off much of that tail, which is why 13% and not more of the sampled lists break.

### 10.2 Rare things with one token each

Section 8 found that the less often a thing was seen in training, the less often its word gets written. With one
token per thing, every name has the same length, so if the many-token names were the trouble, rare things should
now do better. The same table as section 8, for all six runs (the things were seen equally often by B2 and B2f6:
the same scenes):

In [26]:
seen["b2f6"] = Counter()
for line in open(ML / "data/processed/buyers_b2_f6/train.jsonl"):
    r = json.loads(line)
    if r["task"] == "imagine":
        seen["b2f6"].update(scene_things(r["target"]))
print("B2 and B2f6 saw each thing equally often:", seen["b2"] == seen["b2f6"])

for name in ("b1_64", "b2_64", "b2f6_64", "b1_forced", "b2_forced", "b2f6_forced"):
    used, total, unused_at = Counter(), Counter(), [0, 0, 0]
    for r in probe[name]:
        if "|" not in THING.sub("", r["model"]):          # no bar between the list and the line
            continue
        head, line = r["model"].split(" | ", 1) if " | " in r["model"] else r["model"].split("|", 1)
        things = scene_things(r["model"])
        if len(things) != 3 or (THING.search(head) and THING.sub("", head).strip()):   # not a clean list of three
            continue
        for k, t in enumerate(things):
            b = bucket(seen[name.split("_")[0]][t])
            total[b] += 1
            if _named(t, line):
                used[b] += 1
            else:
                unused_at[k] += 1
    rates = "  ".join(f"{b}: {used[b]}/{total[b]} {used[b] / total[b]:.0%}" for b in BUCKETS if total[b])
    print(f"{name:12} {rates}   unused 1st/2nd/3rd {unused_at}")

B2 and B2f6 saw each thing equally often: True
b1_64        never: 1/2 50%  1-9: 17/19 89%  10-29: 49/56 88%  30+: 570/643 89%   unused 1st/2nd/3rd [26, 25, 32]
b2_64        never: 2/6 33%  1-9: 6/8 75%  10-29: 60/94 64%  30+: 434/603 72%   unused 1st/2nd/3rd [76, 70, 63]
b2f6_64      10-29: 20/29 69%  30+: 432/598 72%   unused 1st/2nd/3rd [51, 66, 58]
b1_forced    never: 4/177 2%  1-9: 36/126 29%  10-29: 34/68 50%  30+: 271/352 77%   unused 1st/2nd/3rd [124, 131, 123]


b2_forced    never: 0/3 0%  1-9: 12/51 24%  10-29: 89/149 60%  30+: 344/520 66%   unused 1st/2nd/3rd [100, 93, 85]
b2f6_forced  never: 0/3 0%  1-9: 3/51 6%  10-29: 82/149 55%  30+: 339/520 65%   unused 1st/2nd/3rd [93, 112, 94]


**One token per thing made rare things much worse, not better.** Given the written things, B2f6 uses a thing seen 1–9 times in training only **6%** of the time (3 of 51), against 24% for B2 and 29% for B1; things seen 10–29 times 55% (B2 60%), 30 or more 65% (B2 66%). Choosing its own things, B2f6 never picks a thing seen fewer than 10 times (all 627 of its picks in clean lists were seen at least 10 times) and uses 72% of the common ones, exactly as often as B2. So it is not the length of a name that keeps rare things out of the line; the next cell shows what it is.

Does B2f6 still write from its list? The list-swap test of section 8 (`tools/probe_scene_list.py --sets
runs/eval_sets_b2f6`), where the swap now replaces one thing token by another:

In [27]:
swap["b2f6"] = [json.loads(line) for line in open(F6 / "scene_list/b2f6.jsonl")]
for m in ("b1", "b2", "b2f6"):
    rows = swap[m]
    groups = {}
    for r in rows:
        groups.setdefault((r["set"], bucket(seen[m][r["thing"]])), []).append(r)
    print(f"{m}: {len(rows)} things, mean drop {sum(r['drop'] for r in rows) / len(rows):.2f}")
    for (which, b), g in sorted(groups.items()):
        if len(g) < 10:
            continue
        listed = sorted(math.exp(r["logp_listed"]) for r in g)[len(g) // 2]
        swapped = sorted(math.exp(r["logp_swapped"]) for r in g)[len(g) // 2]
        print(f"   {which} set, seen {b:5} n={len(g):3}  p(word | on the list) {listed:.3f}   "
              f"p(word | swapped out) {swapped:.4f}")

b1: 723 things, mean drop 3.20
   new set, seen 1-9   n=104  p(word | on the list) 0.016   p(word | swapped out) 0.0001
   new set, seen 10-29 n= 38  p(word | on the list) 0.017   p(word | swapped out) 0.0009
   new set, seen 30+   n=138  p(word | on the list) 0.395   p(word | swapped out) 0.0066
   new set, seen never n=176  p(word | on the list) 0.000   p(word | swapped out) 0.0000
   old set, seen 1-9   n= 22  p(word | on the list) 0.626   p(word | swapped out) 0.0028
   old set, seen 10-29 n= 30  p(word | on the list) 0.987   p(word | swapped out) 0.4852
   old set, seen 30+   n=214  p(word | on the list) 0.995   p(word | swapped out) 0.5727
b2: 723 things, mean drop 3.76
   new set, seen 1-9   n= 48  p(word | on the list) 0.183   p(word | swapped out) 0.0001
   new set, seen 10-29 n=126  p(word | on the list) 0.442   p(word | swapped out) 0.0019
   new set, seen 30+   n=279  p(word | on the list) 0.742   p(word | swapped out) 0.0095
   old set, seen 10-29 n= 23  p(word | on the li

**B2f6 still writes from its list**: the word drops by 2.78 nats on average when its thing is swapped out (B2 3.76, B1 3.20). A common thing on the old set is written with probability 0.962 when listed and 0.112 when swapped out, a little more habit than B2 (0.052) and far less than B1 (0.573).

**The rare things of the new set are where it breaks.** With a thing seen 1–9 times on the list, B2f6 gives its word a median probability of **0.012**, against B2's 0.183, fifteen times lower. For things seen 10–29 times B2f6 is better than B2 (0.579 against 0.442), and for things seen 30 times or more about the same (0.781 against 0.742).

This explains section 10.2. With the names in the list, B2 can **copy** a rare thing's word from the list into the line, which a model trained on 1.28 B tokens of text does well whatever the word. With a token, the model has to learn from the scenes which word each token stands for. Ten or more scenes are enough to learn it (better than copying, for 10–29), one to nine are not, and starting the token from its name's own embeddings did not carry the word over.

### 10.3 Does the persona token change what a buyer pictures?

Notebook 6 so far only tested the step from the list to the line. Here is the step from the buyer to the list, in
two ways. First, a correlation: for each picked thing, how much more often the same persona picked it in the
training scenes than all personas did (log of that ratio, "lift"; above 0 means more than average). The written
sales of the validation sets are the yardstick.

In [28]:
BUYER = re.compile(r"<\|buyer:([^|]+)\|>")
per, overall = {}, Counter()
for line in open(ML / "data/processed/buyers_b2_f6/train.jsonl"):
    r = json.loads(line)
    if r["task"] == "imagine":
        per.setdefault(BUYER.search(r["prompt"])[1], Counter()).update(scene_things(r["target"]))
        overall.update(scene_things(r["target"]))
size, n_all = len(overall) + 50, sum(overall.values())
persona_of = {}
for which in ("old", "new"):
    for i, line in enumerate(open(ML / f"runs/eval_sets_b2f6/val_{which}_format6.jsonl")):
        r = json.loads(line)
        if r["task"] == "imagine":
            persona_of[(which, i)] = (BUYER.search(r["prompt"])[1], scene_things(r["target"]))


def lift(persona, thing):               # with half a count added everywhere, so unseen things are not infinite
    c = per[persona]
    return math.log(((c[thing] + 0.5) / (sum(c.values()) + 0.5 * size)) / ((overall[thing] + 0.5) / (n_all + 0.5 * size)))


def summary(pairs):
    values = [lift(p, t) for p, things in pairs for t in things]
    return f"{len(values)} picks, mean lift {sum(values) / len(values):+.2f}, above average {sum(v > 0 for v in values) / len(values):.0%}"


print(f"{'written sales':10}", summary(persona_of.values()))
for name in ("b1_64", "b2_64", "b2f6_64"):
    print(f"{name:13}", summary((persona_of[(r["set"], r["index"])][0], scene_things(r["model"]))
                                for r in probe[name] if r.get("readable")))

written sales 723 picks, mean lift +0.70, above average 70%
b1_64         720 picks, mean lift +1.61, above average 95%
b2_64         719 picks, mean lift +0.65, above average 68%
b2f6_64       684 picks, mean lift +0.81, above average 72%


By this measure the written sales sit at **+0.70** (70% of their picks are things their persona picks more than average). B2 is close to them (+0.65, 68%), B2f6 a little above (+0.81, 72%), and B1 far above (+1.61, 95%): B1 picks almost only the things most typical of each persona, which fits section 8's finding that it writes its commonest things from habit. B2f6's picks are slightly more tied to the persona than B2's, but a correlation cannot say whether the persona token did that; the next test can.

Second, a causal test (`tools/probe_persona_swap.py`): the probability of the written list with the buyer's own
persona, and again with only the persona changed to the next one on the list of 17 (the rest of the card and the
story stay). The drop, per thing, is what the persona itself adds to choosing those things.

In [29]:
pswap = {m: [json.loads(line) for line in open(F6 / f"persona_swap/{m}.jsonl")] for m in ("b1", "b2", "b2f6")}
for m, rows in pswap.items():
    for which in ("old", "new"):
        d = sorted(r["drop_per_thing"] for r in rows if r["set"] == which)
        print(f"{m:5} {which} set: {len(d)} scenes, drop per thing mean {sum(d) / len(d):.2f}, median {d[len(d) // 2]:.2f}, "
              f"lower with the other persona {sum(x > 0 for x in d) / len(d):.0%}")

b1    old set: 89 scenes, drop per thing mean 0.99, median 0.71, lower with the other persona 93%
b1    new set: 152 scenes, drop per thing mean -0.08, median -0.17, lower with the other persona 43%
b2    old set: 89 scenes, drop per thing mean 0.92, median 0.80, lower with the other persona 92%
b2    new set: 152 scenes, drop per thing mean 0.25, median 0.19, lower with the other persona 70%
b2f6  old set: 89 scenes, drop per thing mean 0.74, median 0.46, lower with the other persona 92%
b2f6  new set: 152 scenes, drop per thing mean 0.14, median 0.10, lower with the other persona 72%


**All three models use the persona to choose things on the old set**: with another persona the written list gets less likely in 92–93% of the scenes, by 0.99 (B1), 0.92 (B2) and 0.74 (B2f6) nats per thing on average. **On the new set the persona matters much less**: B2 0.25 and B2f6 0.14 nats per thing (70% and 72% of scenes lower), and for B1 not at all (−0.08; with another persona the list is as likely or more in 57% of the scenes), which fits a model that never saw these sales' things.

So the persona token did not make the step from the buyer to the list stronger; B2f6's persona carries a little less than B2's written persona on both sets. A plausible reason, not tested: the written persona gives the model its words with everything pre-training taught about them, while the token starts as their average and has about 120 training scenes per persona to learn from. The new sales' scenes also depend on far more than the persona (the buyer's own details on the card and in the greeting), which is why the persona alone moves them less.

### 10.4 Reactions and hearing the seller

Format 6 changed only how the persona and the scene's things are written, so reactions should be no worse. Section 3
has the scores. Section 9 and notebook 4 also found that both models misread the seller ("fully vaccinated" heard as
"no vaccinations", "three years old" as two, five or seven). `tools/probe_misread.py` samples five reactions to every
prompt whose latest answer says the vaccines are done or gives the cat's age, and flags a line that says "no
vaccines" or names an age the seller did not say:

In [30]:
mis = {m: [json.loads(line) for line in open(F6 / f"misread/{m}.jsonl")] for m in ("b1", "b2", "b2f6")}
reviewed = json.load(open(F6 / "misread/reviewed.json"))      # every flagged line, read by hand
for m, rows in mis.items():
    for k in ("vaccines", "age"):
        got = [r for r in rows if r["kind"] == k]
        flagged = [r for r in got if r["misread"]]
        real = [r for r in flagged if reviewed[f"{m}/{r['set']}/{r['index']}/{r['sample']}"]["misread"] is True]
        print(f"{m:5} {k:8} {len(got)} reactions to {len({(r['set'], r['index']) for r in got})} prompts: "
              f"{len(flagged)} flagged, {len(real)} misread when read ({len(real) / len(got):.1%}), "
              f"in {len({(r['set'], r['index']) for r in real})} prompts")
print()
for key, v in reviewed.items():
    print(f"{key:22} {'misread' if v['misread'] else 'unclear' if v['misread'] is None else 'not one':8} {v['why']}")
    print(f"{'':22} {v['line'][:105]}")

b1    vaccines 140 reactions to 28 prompts: 7 flagged, 6 misread when read (4.3%), in 2 prompts
b1    age      245 reactions to 49 prompts: 6 flagged, 6 misread when read (2.4%), in 6 prompts
b2    vaccines 140 reactions to 28 prompts: 8 flagged, 8 misread when read (5.7%), in 4 prompts
b2    age      245 reactions to 49 prompts: 2 flagged, 1 misread when read (0.4%), in 1 prompts
b2f6  vaccines 140 reactions to 28 prompts: 3 flagged, 2 misread when read (1.4%), in 2 prompts
b2f6  age      245 reactions to 49 prompts: 3 flagged, 1 misread when read (0.4%), in 1 prompts

b1/old/14/1            misread  the seller says fully vaccinated; the buyer says no vaccinations
                       - | No vaccinated, no fuss and no vet check last month mean she can never join my breeding program, so I'
b1/old/14/2            misread  the seller says fully vaccinated; the buyer says no vaccinations
                       - | No vaccinated and no vet check last month means I can never show or breed

Every flagged line was read (the list above); three were not misreads and two are unclear, so only the confirmed ones are counted. **B1** misreads 6 of 140 reactions to "fully vaccinated" (in 2 of the 28 prompts) and 6 of 245 ages (in 6 of 49 prompts: two months as three, eight years as two or four, three years as two). **B2** misreads 8 of 140 vaccine reactions (4 prompts) and 1 age ("Eight years old is two years old"). **B2f6** misreads 2 of 140 vaccine reactions (2 prompts) and 1 age (eight years as two).

So B2f6 does not hear the seller worse, and it misreads the fewest here, but the numbers are small and the vaccine misreads cluster in a few prompts (the answer "FIV pos but fully vaccinated" is behind most of them), so this is no evidence that format 6 hears better. Format 6 changed nothing about how answers are read, so no difference was expected; B2 and B2f6 still both make the mistake B1 made.

## Version info

The files this notebook read, so the numbers can be traced back.

In [31]:
for f in ["runs/sft_b2/best.pt", "data/processed/buyers_b2/train.jsonl", "data/processed/buyers_b2/val.jsonl",
          "runs/eval_sets_b2/val_old_format5.jsonl", "runs/eval_sets_b2/val_new_format5.jsonl",
          "runs/sft_b1/eval_v2_no_budget.json", "runs/sft_b2/eval/b2_val_old.json",
          "runs/sft_b2/eval/b1_val_new.json", "runs/sft_b2/eval/b2_val_new.json",
          "runs/sft_b2f6/best.pt", "data/processed/buyers_b2_f6/train.jsonl", "data/processed/buyers_b2_f6/val.jsonl",
          "runs/eval_sets_b2f6/val_old_format6.jsonl", "runs/eval_sets_b2f6/val_new_format6.jsonl",
          "runs/sft_b2f6/eval/b2f6_val_old.json", "runs/sft_b2f6/eval/b2f6_val_new.json"]:
    print(hashlib.sha256((ML / f).read_bytes()).hexdigest()[:16], f)

a8df01c605a09522 runs/sft_b2/best.pt
6043ee1a47a79cc6 data/processed/buyers_b2/train.jsonl
16f7326c10afd42a data/processed/buyers_b2/val.jsonl
3e52ea5848e29038 runs/eval_sets_b2/val_old_format5.jsonl
c319c04a9beb8736 runs/eval_sets_b2/val_new_format5.jsonl
5729ae5334185c4f runs/sft_b1/eval_v2_no_budget.json
ece821a22caa0c02 runs/sft_b2/eval/b2_val_old.json
629d9024f070e3df runs/sft_b2/eval/b1_val_new.json
21716beb3fe3f8fc runs/sft_b2/eval/b2_val_new.json


0d65462db2b31dc5 runs/sft_b2f6/best.pt
71225ea1eb715d38 data/processed/buyers_b2_f6/train.jsonl
ab063c6923c719db data/processed/buyers_b2_f6/val.jsonl
386d33dc98981e43 runs/eval_sets_b2f6/val_old_format6.jsonl
55724b6ffc720ffc runs/eval_sets_b2f6/val_new_format6.jsonl
06ebd294b52f102f runs/sft_b2f6/eval/b2f6_val_old.json
950a9117baf9ef17 runs/sft_b2f6/eval/b2f6_val_new.json


The B2 data files were written at 16:01 on 2026-10-12, a minute before the Modal run uploaded them, so these hashes are the data B2 trained on. On 2026-10-13 both B2's and B2f6's data were rebuilt from the snapshot of the 150 new sales (`data/written_sales/sales_b2_150.jsonl`) and came out identical byte for byte. `docs/model-versions.md` has the B1, B2 and B2f6 entries with these hashes.